# Top 50 YouTube Channels by Subscribers

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import warnings, os

warnings.filterwarnings('ignore', category=UserWarning)
plt.style.use('dark_background')

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import glob

csv_path = glob.glob('/kaggle/input/**/*.csv', recursive=True)[0]
df = pd.read_csv(csv_path)

def parse_count(val):
    if pd.isna(val): return 0
    val = str(val).strip()
    if val.endswith('B'): return float(val[:-1]) * 1e9
    if val.endswith('M'): return float(val[:-1]) * 1e6
    if val.endswith('K'): return float(val[:-1]) * 1e3
    try: return float(val)
    except: return 0

import re
df['channel'] = df['channel'].apply(lambda x: re.sub(r'[^\x00-\x7F]+', '', str(x)).strip() or '???')
df['sub_num'] = df['subscribers'].apply(parse_count)
df['views_num'] = df['video_views'].apply(parse_count)
df['uploads_num'] = df['uploads'].apply(parse_count)
df.head()

In [ ]:
colors = plt.cm.hot(np.linspace(0.3, 0.85, 50))

fig, ax = plt.subplots(figsize=(14, 18))
bars = ax.barh(range(len(df)), df['sub_num'], color=colors[::-1], edgecolor='none', height=0.7)

ax.set_yticks(range(len(df)))
ax.set_yticklabels(df['channel'], fontsize=9, color='#e0e0e0')
ax.invert_yaxis()
ax.set_title('Top 50 YouTube Channels by Subscribers', fontsize=16, fontweight='bold', pad=15, color='white')

for i, (v, s) in enumerate(zip(df['sub_num'], df['subscribers'])):
    ax.text(v + max(df['sub_num'])*0.005, i, s, va='center', fontsize=8, color='#ffffff')

ax.spines[['top', 'right', 'bottom']].set_visible(False)
ax.xaxis.set_visible(False)
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

In [ ]:
active = df[df['views_num'] > 0].copy()

fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(
    active['sub_num'] / 1e6,
    active['views_num'] / 1e9,
    s=active['uploads_num'] / max(active['uploads_num']) * 400 + 20,
    c=active['sub_num'], cmap='plasma', alpha=0.85, edgecolors='#333', linewidth=0.5
)

for _, row in active.head(10).iterrows():
    ax.annotate(row['channel'], (row['sub_num']/1e6, row['views_num']/1e9),
                fontsize=7, color='#e0e0e0', textcoords='offset points', xytext=(5, 5))

ax.set_xlabel('Subscribers (M)', fontsize=11, color='#ccc')
ax.set_ylabel('Total Views (B)', fontsize=11, color='#ccc')
ax.set_title('Subscribers vs Views\n(bubble size = uploads)', fontsize=14, fontweight='bold', color='white')
ax.spines[['top', 'right']].set_visible(False)
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()